In [136]:
import sys
sys.path.append("../src/")
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [137]:
from relaxers.quadratic_model_lin_approx import quadratic_model_lin_approx # type: ignore[reportMissingImports]
from relaxers.quadratic_lin_approx_no_surrogate import quadratic_lin_approx_no_surrogate # type: ignore[reportMissingImports]
from relaxers.quadratic_lin_approx_nosur_refine import quadratic_lin_approx_no_surrogate_refine # type: ignore[reportMissingImports]
from testers.classical_solve_lin_program import solve_lp_return_x # type: ignore[reportMissingImports]
from testers.classical_solve_quad_program import solve_qcp # type: ignore[reportMissingImports]
import matplotlib.pyplot as plt
import numpy as np

In [138]:
LIN_APPROX = 100

# Whether to do an inner or outer approximation
OUTER_APPROXIMATION = True

# Whether the coefficient is part of the surrogate
COEFFICIENT_SURROGATE = True

# Whether to bound the surrogates with bound constraints.
SURROGATE_BOUND_BELOW = True
SURROGATE_BOUND_ABOVE = True

# Whether to divide by lambda in an inner approximation
REMOVE_DIVISION = False

In [139]:
# Put the original QCP into gurobi
qcp_value, qcp_time = solve_qcp(f_name="model11_quad_reform_2.json", verbose=True)
print(qcp_value, qcp_time)

phi[pipe01_entry01_entry03]: 159.99999999999997
phi[pipe02_N01_N02]: 146.41016151850428
phi[pipe03_entry02_N03]: 140.0
phi[pipe04_N02_exit01]: 100.0
phi[pipe05_N02_N04]: 46.41016152137879
phi[pipe06_N03_N04]: 153.58983848132388
phi[pipe07_N05_exit02]: 120.0
phi[pipe08_N05_exit03]: 80.0
phi[CS01_entry03_N01]: 160.00000000000057
phi[CS02_N04_N05]: 200.0
psi[N01]: 2868.535990757481
psi[N02]: 2361.430301534333
psi[N04]: 2310.4758420050157
psi[N05]: 3751.4035340100886
psi[entry01]: 4240.0
psi[entry02]: 3332.209314082521
psi[entry03]: 1856.2773914427376
psi[exit01]: 2124.86227937257
psi[exit02]: 3410.74558214849
psi[exit03]: 3599.999999707233
4692.145737713451 0.002000093460083008


In [140]:
quadratic_lin_approx_no_surrogate(LIN_APPROX, OUTER_APPROXIMATION, remove_division=REMOVE_DIVISION,
                            f_name="model11_quad_reform_nofix.json")
val, time, x = solve_lp_return_x()
print(x)
quad, q_time = solve_qcp(f_name="model11_quad_reform_nofix.json")

print(f"Error: {val - quad}")

[160.00000000000003, 146.41577060931903, 140.0, 100.0, 46.41577060931898, 153.584229390681, 120.0, 80.0, 160.00000000000003, 200.0, 2472.5063217220954, 1965.4537153315641, 1914.5788718810093, 3108.605515599187, 2205.146841577046, 2935.7123500720927, 1600.0, 1728.8856932864835, 2768.064387568861, 2957.318805204926]
Error: -1.4336252510511258


In [ ]:
# In this case there are 10 quadratic variables
# Define a set of functions for each

# TODO: It should intelligently search for the quadratic variables, not just assume they're at the start
# TODO: Reduce the number of other approximators as it goes on
# TODO: Analyze condition number of the new system
# TODO: Try bunching around the point (move the other points towards it)

iters = 1

for _ in range(iters):
    quads = x[:10]
    points_functions = []


    def np_uniform_add_q_factory(q):
        def np_uniform_add_q(lower, upper, num):
                uniform = np.linspace(lower, upper, num)
                idx = np.searchsorted(uniform, q)
                return np.insert(uniform, idx, q)
        return np_uniform_add_q

    for q in quads:
        points_functions.append(np_uniform_add_q_factory(q))

    points_functions[0](0, 200, 10)

    quadratic_lin_approx_no_surrogate_refine(LIN_APPROX, OUTER_APPROXIMATION, points_function=points_functions,
                                                remove_division=REMOVE_DIVISION, f_name="model11_quad_reform_nofix.json")
    val, time, x = solve_lp_return_x()
    quad, q_time = solve_qcp(f_name="model11_quad_reform_nofix.json")

print(x)
print(f"Error: {val - quad}")

[160.00000000000003, 146.41016151377545, 140.0, 100.0, 46.41016151377543, 153.58983848622458, 120.0, 80.0, 160.00000000000003, 200.0, 2472.5063217220954, 1965.4006380260364, 1914.4461786171896, 3108.390068213942, 2205.6141364354066, 2936.1796449304534, 1600.0, 1728.8326159809558, 2767.732116469026, 2956.9865341050904]
Error: -1.5775094652781263e-06
